In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

df1 = pd.read_csv('./data/CDC Diabetes Dataset.csv')

In [ ]:
# Keep only non-diabetic (0) and diabetic (2)
df = df1[df1['Diabetes_012'].isin([0, 2])].copy()

# 1) columns for ARM
arm_cols_binary = [
    'HighBP','HighChol','CholCheck','Smoker','Stroke','HeartDiseaseorAttack',
    'PhysActivity','Fruits','Veggies','HvyAlcoholConsump',
    'AnyHealthcare','NoDocbcCost','DiffWalk','Sex'
]

arm_cols_ordinal = ['BMI','Age','GenHlth','PhysHlth','MentHlth','Education','Income']

# 2) binning
df_arm = df[arm_cols_binary + arm_cols_ordinal].copy()

df_arm['BMI_bin'] = pd.cut(df_arm['BMI'], bins=[0, 18.5, 25, 30, 100], 
                           labels=['BMI_Under','BMI_Normal','BMI_Over','BMI_Obese'])
df_arm['Age_bin'] = pd.cut(df_arm['Age'], bins=[0, 4, 7, 10, 14], 
                           labels=['Age_Young','Age_Mid','Age_Older','Age_Oldest'])

# self-reported days: 0 / 1-13 / 14-30 (common shreshold)
df_arm['PhysHlth_bin'] = pd.cut(df_arm['PhysHlth'], bins=[-1, 0, 13, 30],
                                labels=['PhysHlth_0','PhysHlth_1_13','PhysHlth_14_30'])
df_arm['MentHlth_bin'] = pd.cut(df_arm['MentHlth'], bins=[-1, 0, 13, 30],
                                labels=['MentHlth_0','MentHlth_1_13','MentHlth_14_30'])

# GenHlth: 1-5
df_arm['GenHlth_bin'] = df_arm['GenHlth'].map({
    1:'GenHlth_Excellent',2:'GenHlth_VeryGood',3:'GenHlth_Good',4:'GenHlth_Fair',5:'GenHlth_Poor'
})

# Education/Income interprete into binary
df_arm['Edu_bin'] = np.where(df_arm['Education'] >= 4, 'Edu_High', 'Edu_Low')
df_arm['Income_bin'] = np.where(df_arm['Income'] >= 6, 'Income_High', 'Income_Low')

# 3) binary columns: 1 -> column name
for c in arm_cols_binary:
    df_arm[c] = df_arm[c].where(df_arm[c] == 1, pd.NA)
    df_arm[c] = df_arm[c].replace(1, c)

# 4) final transaction dataset for ARM
item_cols = arm_cols_binary + ['BMI_bin','Age_bin','PhysHlth_bin','MentHlth_bin','GenHlth_bin','Edu_bin','Income_bin']
transactions = df_arm[item_cols]


DTypePromotionError: The DType <class 'numpy.dtypes.StrDType'> could not be promoted by <class 'numpy.dtypes._PyFloatDType'>. This means that no common DType exists for the given inputs. For example they cannot be stored in a single array unless the dtype is `object`. The full list of DTypes is: (<class 'numpy.dtypes.StrDType'>, <class 'numpy.dtypes._PyFloatDType'>)

In [ ]:
# 每行转为item list
tx = transactions.apply(lambda row: [x for x in row.dropna().tolist()], axis=1).tolist()

te = TransactionEncoder()
te_ary = te.fit(tx).transform(tx)
df_onehot = pd.DataFrame(te_ary, columns=te.columns_)

# frequent itemsets
freq = apriori(df_onehot, min_support=0.02, use_colnames=True)  # support阈值按你数据调整
freq = freq.sort_values('support', ascending=False)

# rules
rules = association_rules(freq, metric='lift', min_threshold=1.2)
rules = rules.sort_values(['lift','confidence','support'], ascending=False)
rules.head(20)


In [ ]:
strong_rules = rules[
    (rules["lift"] > 2.0) &
    (rules["confidence"] > 0.4)
]

strong_rules[["antecedents", "consequents", "support", "confidence", "lift"]]